<a href="https://colab.research.google.com/github/VarshaP-0405/NLP-Skill-Hometask/blob/main/HMM%20POS%20Tagger.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
!pip install conllu requests

In [3]:
import math
from collections import Counter, defaultdict
import requests
import conllu

TRAIN_URL = "https://raw.githubusercontent.com/UniversalDependencies/UD_English-EWT/master/en_ewt-ud-train.conllu"
TEST_URL = "https://raw.githubusercontent.com/UniversalDependencies/UD_English-EWT/master/en_ewt-ud-test.conllu"

train_data = requests.get(TRAIN_URL).text
test_data = requests.get(TEST_URL).text

train_sentences = conllu.parse(train_data)
test_sentences = conllu.parse(test_data)

print("Training sentences:", len(train_sentences))
print("Testing sentences:", len(test_sentences))
train_pairs = []

for sentence in train_sentences:
    pairs = []

    for token in sentence:
        # Ignore multi-word tokens such as 1-2
        if isinstance(token["id"], int):
            word = token["form"].lower()
            tag = token["upos"]

            pairs.append((word, tag))

    if pairs:
        train_pairs.append(pairs)
transition_counts = defaultdict(Counter)
tag_counts = Counter()

START = "<START>"
END = "<END>"

for sentence in train_pairs:
    previous_tag = START

    for word, tag in sentence:
        transition_counts[previous_tag][tag] += 1
        tag_counts[tag] += 1
        previous_tag = tag

    transition_counts[previous_tag][END] += 1
transition_probs = defaultdict(dict)

for previous_tag, next_tags in transition_counts.items():

    total = sum(next_tags.values())

    for next_tag, count in next_tags.items():
        transition_probs[previous_tag][next_tag] = math.log(
            count / total
        )
emission_counts = defaultdict(Counter)

for sentence in train_pairs:
    for word, tag in sentence:
        emission_counts[tag][word] += 1


emission_probs = defaultdict(dict)

for tag, words in emission_counts.items():

    total = sum(words.values())

    for word, count in words.items():
        emission_probs[tag][word] = math.log(
            count / total
        )
tags = list(tag_counts.keys())
SMOOTH = math.log(1e-10)
def viterbi(words):

    words = [word.lower() for word in words]

    # Viterbi table
    viterbi_table = []
    backpointer = []
    first_word = words[0]

    current_scores = {}
    current_backpointer = {}

    for tag in tags:

        transition = transition_probs[START].get(tag, SMOOTH)

        emission = emission_probs[tag].get(
            first_word,
            SMOOTH
        )

        current_scores[tag] = transition + emission
        current_backpointer[tag] = None

    viterbi_table.append(current_scores)
    backpointer.append(current_backpointer)
    for i in range(1, len(words)):

        word = words[i]

        current_scores = {}
        current_backpointer = {}

        for current_tag in tags:

            emission = emission_probs[current_tag].get(
                word,
                SMOOTH
            )

            best_score = float("-inf")
            best_previous_tag = None

            for previous_tag in tags:

                transition = transition_probs[
                    previous_tag
                ].get(
                    current_tag,
                    SMOOTH
                )

                score = (
                    viterbi_table[i - 1][previous_tag]
                    + transition
                    + emission
                )

                if score > best_score:
                    best_score = score
                    best_previous_tag = previous_tag

            current_scores[current_tag] = best_score
            current_backpointer[current_tag] = best_previous_tag

        viterbi_table.append(current_scores)
        backpointer.append(current_backpointer)
    best_final_tag = None
    best_final_score = float("-inf")

    for tag in tags:

        end_transition = transition_probs[tag].get(
            END,
            SMOOTH
        )

        score = viterbi_table[-1][tag] + end_transition

        if score > best_final_score:
            best_final_score = score
            best_final_tag = tag
    predicted_tags = [best_final_tag]

    for i in range(len(words) - 1, 0, -1):

        previous_tag = backpointer[i][predicted_tags[-1]]

        predicted_tags.append(previous_tag)

    predicted_tags.reverse()

    return predicted_tags
total_words = 0
correct_words = 0

tag_correct = Counter()
tag_total = Counter()

for sentence in test_sentences:

    words = []
    actual_tags = []

    for token in sentence:

        if isinstance(token["id"], int):

            words.append(token["form"].lower())
            actual_tags.append(token["upos"])

    if not words:
        continue

    predicted_tags = viterbi(words)

    for actual, predicted in zip(
        actual_tags,
        predicted_tags
    ):

        total_words += 1
        tag_total[actual] += 1

        if actual == predicted:
            correct_words += 1
            tag_correct[actual] += 1
accuracy = (correct_words / total_words) * 100

print("\n====================================")
print("HMM POS TAGGER EVALUATION REPORT")
print("====================================")

print("Total test words :", total_words)
print("Correct predictions:", correct_words)
print("Incorrect predictions:", total_words - correct_words)

print(f"POS Tagging Accuracy: {accuracy:.2f}%")

print("\nPer-tag Accuracy")
print("------------------------------------")

for tag in sorted(tag_total):

    tag_accuracy = (
        tag_correct[tag] / tag_total[tag]
    ) * 100

    print(
        f"{tag:8s} : "
        f"{tag_accuracy:.2f}% "
        f"({tag_correct[tag]}/{tag_total[tag]})"
    )
sentence = input(
    "\nEnter a sentence: "
)

words = sentence.split()

predicted_tags = viterbi(words)


print("\nPOS Tagging Result")
print("------------------")

for word, tag in zip(words, predicted_tags):

    print(f"{word} → {tag}")
print("\nExample:")
print("The → DET")
print("student → NOUN")
print("reads → VERB")
print("a → DET")
print("book → NOUN")

Training sentences: 12544
Testing sentences: 2077

HMM POS TAGGER EVALUATION REPORT
Total test words : 25094
Correct predictions: 22291
Incorrect predictions: 2803
POS Tagging Accuracy: 88.83%

Per-tag Accuracy
------------------------------------
ADJ      : 87.86% (1571/1788)
ADP      : 93.58% (1895/2025)
ADV      : 84.89% (1011/1191)
AUX      : 97.15% (1499/1543)
CCONJ    : 98.91% (728/736)
DET      : 97.36% (1847/1897)
INTJ     : 78.51% (95/121)
NOUN     : 89.91% (3707/4123)
NUM      : 62.73% (340/542)
PART     : 94.61% (614/649)
PRON     : 96.86% (2096/2164)
PROPN    : 54.46% (1130/2075)
PUNCT    : 98.77% (3058/3096)
SCONJ    : 72.66% (279/384)
SYM      : 84.07% (95/113)
VERB     : 89.25% (2325/2605)
X        : 2.38% (1/42)

Enter a sentence: The Student reads a book

POS Tagging Result
------------------
The → DET
Student → NOUN
reads → VERB
a → DET
book → NOUN

Example:
The → DET
student → NOUN
reads → VERB
a → DET
book → NOUN
